# 🧠 Phase 4 — Model Development, Benchmarking & Reporting

> **Malicious PDF Detector** — Training 4 classifiers and selecting the champion model

This notebook trains and evaluates four classification models:
1. **Random Forest** — ensemble of decision trees with GridSearchCV
2. **XGBoost** — gradient-boosted trees (CPU-optimized via `tree_method='hist'`)
3. **LightGBM** — histogram-based gradient boosting
4. **MLP (PyTorch)** — 3-layer neural network with early stopping

All models are evaluated on Accuracy, F1, Precision, Recall, AUC-ROC and compared side-by-side.

In [ ]:
# === Cell 1: Setup & Data Loading ===
import sys
import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

warnings.filterwarnings('ignore')

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.config import (
    FEATURE_COLUMNS, PROCESSED_DATA_DIR, TRAINED_MODELS_DIR,
    FIGURES_DIR, RESULTS_DIR, RANDOM_SEED, MODEL_CONFIGS
)
from src.utils.logger import get_logger
from sklearn.datasets import make_classification

logger = get_logger('notebook.04_model_training')

print('Phase 4: Model Development, Benchmarking & Reporting')
print('='*55)

# Load data — use real data if available, otherwise synthetic
train_path = PROCESSED_DATA_DIR / 'train.csv'
val_path = PROCESSED_DATA_DIR / 'val.csv'
test_path = PROCESSED_DATA_DIR / 'test.csv'

if train_path.exists() and val_path.exists() and test_path.exists():
    print('Loading real dataset from processed CSVs...')
    train_df = pd.read_csv(train_path)
    val_df = pd.read_csv(val_path)
    test_df = pd.read_csv(test_path)
    
    label_col = 'Class' if 'Class' in train_df.columns else train_df.columns[-1]
    feature_cols = [c for c in FEATURE_COLUMNS if c in train_df.columns]
    
    X_train = train_df[feature_cols].values
    y_train = train_df[label_col].values
    X_val = val_df[feature_cols].values
    y_val = val_df[label_col].values
    X_test = test_df[feature_cols].values
    y_test = test_df[label_col].values
    data_source = 'CIC PDFMal2022'
else:
    print('Real dataset not found. Using synthetic data for demonstration...')
    np.random.seed(RANDOM_SEED)
    X, y = make_classification(
        n_samples=2000, n_features=37, n_informative=15,
        n_redundant=5, n_clusters_per_class=2,
        weights=[0.6, 0.4], random_state=RANDOM_SEED,
    )
    X_train, X_val, X_test = X[:1400], X[1400:1700], X[1700:]
    y_train, y_val, y_test = y[:1400], y[1400:1700], y[1700:]
    feature_cols = FEATURE_COLUMNS
    data_source = 'Synthetic (demo)'

print(f'Data Source: {data_source}')
print(f'Train: {X_train.shape} | Val: {X_val.shape} | Test: {X_test.shape}')
print(f'Features: {len(feature_cols)}')
print(f'Train class dist: {dict(zip(*np.unique(y_train, return_counts=True)))}')

In [ ]:
# === Cell 2: Train All 4 Models ===
import time
from src.models.baseline import BaselineModel
from src.models.mlp import MaliciousPDFClassifier, create_data_loaders, train_mlp

print('='*60)
print('TRAINING ALL MODELS')
print('='*60)

trained_models = {}
training_times = {}
cv_scores = {}

# --- Tree-based models ---
for model_type in ['random_forest', 'xgboost', 'lightgbm']:
    print(f'\nTraining {model_type}...')
    model = BaselineModel(model_type)
    model.train(X_train, y_train, X_val, y_val)
    model.save()
    trained_models[model_type] = model
    training_times[model_type] = model.training_time_sec
    cv_scores[model_type] = model.cv_results_['best_score']
    print(f'  Done: {model.training_time_sec:.2f}s, CV F1={model.cv_results_["best_score"]:.4f}')

# --- MLP ---
print('\nTraining MLP (PyTorch)...')
mlp = MaliciousPDFClassifier(input_dim=len(feature_cols))
print(f'  Parameters: {sum(p.numel() for p in mlp.parameters()):,}')

train_loader, val_loader = create_data_loaders(
    X_train.astype(np.float32), y_train.astype(np.float32),
    X_val.astype(np.float32), y_val.astype(np.float32),
    batch_size=64
)

mlp_history = train_mlp(mlp, train_loader, val_loader, max_epochs=100, patience=10)
trained_models['mlp'] = mlp
training_times['mlp'] = mlp_history['training_time_sec']

print(f'\n  MLP Done: {mlp_history["epochs_trained"]} epochs, '
      f'{mlp_history["training_time_sec"]:.2f}s, '
      f'Best val loss={mlp_history["best_val_loss"]:.4f}')

# Summary
print('\n' + '='*60)
print('TRAINING SUMMARY')
print('='*60)
for name, t in training_times.items():
    extra = f', CV F1={cv_scores[name]:.4f}' if name in cv_scores else ''
    print(f'  {name}: {t:.2f}s{extra}')

In [ ]:
# === Cell 3: Hyperparameter Tuning Results ===

print('='*60)
print('HYPERPARAMETER TUNING RESULTS')
print('='*60)

for model_type in ['random_forest', 'xgboost', 'lightgbm']:
    model = trained_models[model_type]
    print(f'\n--- {model.display_name} ---')
    print(f'  Best params: {model.best_params_}')
    print(f'  Best CV F1:  {model.cv_results_["best_score"]:.4f}')
    print(f'  Training time: {model.training_time_sec:.2f}s')
    
    # Show top-5 grid search results
    params_list = model.cv_results_['params']
    scores = model.cv_results_['mean_test_score']
    stds = model.cv_results_['std_test_score']
    sorted_idx = np.argsort(scores)[::-1][:5]
    
    print(f'  Top-5 configurations:')
    for rank, idx in enumerate(sorted_idx, 1):
        print(f'    #{rank}: F1={scores[idx]:.4f} (+/- {stds[idx]:.4f}) | {params_list[idx]}')

In [ ]:
# === Cell 4: MLP Training Curves ===

from src.utils.visualization import _apply_dark_theme, COLORS, GRADIENT_PALETTE, _save_figure

_apply_dark_theme()

fig, axes = plt.subplots(1, 3, figsize=(20, 6))
fig.suptitle('MLP Training Curves', fontsize=16, fontweight='bold', y=1.02)

epochs = range(1, mlp_history['epochs_trained'] + 1)

# Loss curves
ax = axes[0]
ax.plot(epochs, mlp_history['train_loss'], color=COLORS['accent'], linewidth=2, label='Train Loss')
ax.plot(epochs, mlp_history['val_loss'], color=COLORS['malicious'], linewidth=2, label='Val Loss')
ax.axvline(mlp_history['best_epoch'] + 1, color=COLORS['benign'], linestyle='--', alpha=0.7, label='Best Epoch')
ax.set_xlabel('Epoch')
ax.set_ylabel('BCE Loss')
ax.set_title('Training & Validation Loss')
ax.legend()

# Accuracy curve
ax = axes[1]
ax.plot(epochs, mlp_history['val_accuracy'], color=COLORS['benign'], linewidth=2, label='Val Accuracy')
ax.axhline(max(mlp_history['val_accuracy']), color=COLORS['accent'], linestyle='--', alpha=0.5,
           label=f'Peak: {max(mlp_history["val_accuracy"]):.4f}')
ax.set_xlabel('Epoch')
ax.set_ylabel('Accuracy')
ax.set_title('Validation Accuracy')
ax.legend()

# F1 curve
ax = axes[2]
ax.plot(epochs, mlp_history['val_f1'], color='#F59E0B', linewidth=2, label='Val F1')
ax.axhline(max(mlp_history['val_f1']), color=COLORS['accent'], linestyle='--', alpha=0.5,
           label=f'Peak: {max(mlp_history["val_f1"]):.4f}')
ax.set_xlabel('Epoch')
ax.set_ylabel('F1 Score')
ax.set_title('Validation F1 Score')
ax.legend()

plt.tight_layout()
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
_save_figure(fig, str(FIGURES_DIR / 'mlp_training_curves.png'))
plt.show()

print(f'\nMLP Summary:')
print(f'  Epochs trained: {mlp_history["epochs_trained"]}')
print(f'  Best epoch: {mlp_history["best_epoch"] + 1}')
print(f'  Best val loss: {mlp_history["best_val_loss"]:.4f}')
print(f'  Peak val accuracy: {max(mlp_history["val_accuracy"]):.4f}')
print(f'  Peak val F1: {max(mlp_history["val_f1"]):.4f}')

In [ ]:
# === Cell 5: Test Set Evaluation ===

from src.models.evaluator import evaluate_model, ModelEvaluator

print('='*60)
print('TEST SET EVALUATION')
print('='*60)

evaluator = ModelEvaluator()
all_results = evaluator.evaluate_all(trained_models, X_test, y_test)

for r in all_results:
    print(f'\n--- {r["model_name"]} ---')
    print(f'  Accuracy:  {r["accuracy"]:.4f}')
    print(f'  F1-Score:  {r["f1"]:.4f}')
    print(f'  Precision: {r["precision"]:.4f}')
    print(f'  Recall:    {r["recall"]:.4f}')
    print(f'  AUC-ROC:   {r["auc_roc"]:.4f}')
    print(f'  Inference: {r["inference_time_ms"]:.1f} ms')

In [ ]:
# === Cell 6: Model Comparison Table ===

print('='*60)
print('MODEL COMPARISON (sorted by F1-Score)')
print('='*60)

comparison = evaluator.get_comparison()
display(comparison.style.background_gradient(
    subset=['F1-Score', 'AUC-ROC', 'Accuracy'], cmap='YlGn'
).format(precision=4).set_properties(**{
    'background-color': '#161B22',
    'color': '#E6EDF3',
    'border-color': '#30363D'
}))

best = evaluator.get_best_model()
print(f'\nChampion Model: {best["model_name"]}')
print(f'F1-Score: {best["f1"]:.4f} | AUC-ROC: {best["auc_roc"]:.4f}')

In [ ]:
# === Cell 7: Confusion Matrices (2x2 Grid) ===

from src.utils.visualization import plot_confusion_matrix as plot_cm

_apply_dark_theme()

fig, axes = plt.subplots(2, 2, figsize=(16, 14))
fig.suptitle('Confusion Matrices - All Models', fontsize=18, fontweight='bold', y=1.02)

for idx, result in enumerate(all_results):
    ax = axes[idx // 2, idx % 2]
    cm = result['confusion_matrix']
    total = cm.sum()
    cm_pct = cm / total * 100
    
    import seaborn as sns
    import matplotlib.colors as mcolors
    
    annot = np.empty_like(cm, dtype=object)
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            annot[i, j] = f'{cm[i, j]:,}\n({cm_pct[i, j]:.1f}%)'
    
    cmap = mcolors.LinearSegmentedColormap.from_list(
        'cm_cmap', ['#161B22', '#1B3A5C', '#2563EB', '#7C4DFF']
    )
    
    sns.heatmap(
        cm, annot=annot, fmt='', cmap=cmap,
        xticklabels=['Benign', 'Malicious'],
        yticklabels=['Benign', 'Malicious'],
        linewidths=2, linecolor=COLORS['bg_dark'],
        annot_kws={'size': 12, 'fontweight': 'bold'},
        ax=ax,
    )
    
    ax.set_title(f'{result["model_name"]} (F1={result["f1"]:.4f})',
                 fontsize=12, fontweight='bold', pad=10)
    ax.set_xlabel('Predicted', fontsize=10)
    ax.set_ylabel('Actual', fontsize=10)

plt.tight_layout()
_save_figure(fig, str(FIGURES_DIR / 'confusion_matrices_grid.png'))
plt.show()

In [ ]:
# === Cell 8: ROC Curves (Overlaid) ===

from src.utils.visualization import plot_roc_curves

roc_data = {}
for r in all_results:
    roc_data[r['model_name']] = {
        'fpr': r['fpr'],
        'tpr': r['tpr'],
        'auc': r['auc_roc'],
    }

fig = plot_roc_curves(
    roc_data,
    save_path=str(FIGURES_DIR / 'roc_curves_all.png'),
    title='ROC Curves - Model Comparison'
)
plt.show()

print('\nAUC-ROC Ranking:')
for name, data in sorted(roc_data.items(), key=lambda x: -x[1]['auc']):
    print(f'  {name}: {data["auc"]:.4f}')

In [ ]:
# === Cell 9: Feature Importance (Top 15) ===

from src.utils.visualization import plot_feature_importance

print('='*60)
print('FEATURE IMPORTANCE ANALYSIS')
print('='*60)

# Use the best tree model for importance
tree_models = {k: v for k, v in trained_models.items() if k != 'mlp'}
best_tree_name = max(tree_models, key=lambda k: [
    r['f1'] for r in all_results if r['model_name'] == tree_models[k].display_name
][0])
best_tree = tree_models[best_tree_name]

print(f'\nShowing importance from: {best_tree.display_name}')

# Get feature names (use actual feature names or generic)
if len(feature_cols) == len(best_tree.feature_importances_):
    feat_names = feature_cols
else:
    feat_names = [f'feature_{i}' for i in range(len(best_tree.feature_importances_))]

fig = plot_feature_importance(
    best_tree.feature_importances_,
    feat_names,
    save_path=str(FIGURES_DIR / 'feature_importance_best.png'),
    top_n=15,
    title=f'Top-15 Feature Importance ({best_tree.display_name})'
)
plt.show()

# Print top-10
imp = best_tree.get_feature_importances(feat_names)
print('\nTop-10 Most Important Features:')
for i, (name, score) in enumerate(list(imp.items())[:10], 1):
    bar = '#' * int(score * 200)
    print(f'  {i:2d}. {name:25s} {score:.4f} {bar}')

In [ ]:
# === Cell 10: Model Selection Rationale ===

print('='*70)
print('MODEL SELECTION RATIONALE')
print('='*70)

# Get all results sorted
sorted_results = sorted(all_results, key=lambda r: -r['f1'])

print('\n  Final Rankings (by F1-Score):')
print('  ' + '-'*66)
for i, r in enumerate(sorted_results, 1):
    marker = ' <-- SELECTED' if i == 1 else ''
    print(f'  {i}. {r["model_name"]:20s} | F1={r["f1"]:.4f} | '
          f'AUC={r["auc_roc"]:.4f} | '
          f'Inf={r["inference_time_ms"]:.0f}ms{marker}')

champion = sorted_results[0]

print(f'\n  Champion: {champion["model_name"]}')
print(f'  Reason: Highest F1-Score ({champion["f1"]:.4f}) with strong '
      f'AUC-ROC ({champion["auc_roc"]:.4f})')

print(f'\n  Classification Report ({champion["model_name"]}):')
print(champion['classification_report'])

# Recommendation for Phase 5
print('\n  Phase 5 Recommendation:')
print('  ' + '-'*66)
if 'MLP' in champion['model_name']:
    print('  The MLP model is selected for INT8 post-training quantization')
    print('  in Phase 5. Its PyTorch architecture (Linear->BatchNorm->ReLU')
    print('  blocks) is optimized for fbgemm quantization backend.')
else:
    print(f'  The {champion["model_name"]} model achieves the best F1.')
    print('  For quantization (Phase 5), the MLP will still be quantized')
    print('  as a secondary deployment option for latency-sensitive use cases.')

print(f'\n  All 4 model checkpoints saved to: {TRAINED_MODELS_DIR}')
print(f'  Report figures saved to: {FIGURES_DIR}')
print(f'  Comparison CSV saved to: {RESULTS_DIR / "model_comparison.csv"}')
print('\n  Ready for Phase 5: Quantization & Optimization!')